# Phase 7 — Baseline Model + Bias–Variance Analysis

## H&M Personalized Fashion Recommendations → Customer Purchase Prediction

### Objective

Phase 6 prepared the customer-level feature matrix using a leakage-safe preprocessing pipeline.

In this phase, we establish the **first machine-learning benchmark** for the project.

Our prediction problem is:

> **Predict whether a customer will make at least one purchase during the next 30 days.**

The main goals of Phase 7 are:

1. Establish a naive baseline using `DummyClassifier`.
2. Train a Logistic Regression baseline.
3. Evaluate both models using classification metrics.
4. Compare training and validation performance.
5. Diagnose bias and variance.
6. Analyze the classification threshold.
7. Establish a benchmark for all future models.
8. Save the baseline results and model artifacts.

---

## Why a baseline is important

A sophisticated model is meaningful only if it improves over a simple reference.

For example, if most customers do not purchase in the next 30 days, a model that predicts **0 for everyone** may already achieve high accuracy.

Therefore, we need to answer:

> **Does our ML model actually learn useful purchasing patterns beyond the majority-class baseline?**

This baseline will become the reference point for Phase 8.


## 7.1 Phase 7 Architecture

```text
                    Phase 5
              Temporal Train/Val/Test
                       │
                       ▼
                    Phase 6
              ML Preprocessing
                       │
                       ▼
             ┌────────────────────┐
             │ Processed Features │
             └─────────┬──────────┘
                       │
             ┌─────────┴─────────┐
             ▼                   ▼
      Dummy Classifier     Logistic Regression
             │                   │
             └─────────┬─────────┘
                       ▼
                Model Evaluation
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
       Metrics     Threshold     Bias/Variance
          │            │            │
          └────────────┼────────────┘
                       ▼
              Baseline Benchmark
                       │
                       ▼
                   Phase 8
             Multiple ML Algorithms
```


## 7.2 Important Modeling Principle

We will preserve the temporal split created in Phase 5.

We **will not randomly split the customers again**.

The intended evaluation structure is:

```text
Past customer behavior
        │
        ▼
     TRAIN
        │
        ▼
  Validation period
        │
        ▼
      TEST
```

This better represents the real-world scenario:

> Use historical customer behavior to predict future purchases.

The test set remains untouched during model selection.


## 7.3 Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

RANDOM_STATE = 42

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train_phase5.parquet"
VAL_PATH = PROCESSED_DIR / "validation_phase5.parquet"
TEST_PATH = PROCESSED_DIR / "test_phase5.parquet"

STANDARD_PREPROCESSOR_PATH = MODELS_DIR / "preprocessor_standard_phase6.joblib"

print("Project root:", PROJECT_ROOT.resolve())
print("Models:", MODELS_DIR.resolve())
print("Results:", RESULTS_DIR.resolve())


## 7.4 Load Phase 5 Data

We use the raw Phase 5 customer-level datasets because they contain:

- engineered features,
- target,
- customer IDs.

The preprocessing pipeline from Phase 6 will then be loaded and applied exactly as it was defined there.

This approach also makes the notebook reproducible from the saved project artifacts.


In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(train_df.head())


## 7.5 Separate Features and Target

In [ ]:
TARGET_COL = "target"
ID_COL = "customer_id"

customer_id_train = train_df[ID_COL].copy()
customer_id_val = val_df[ID_COL].copy()
customer_id_test = test_df[ID_COL].copy()

X_train = train_df.drop(columns=[TARGET_COL, ID_COL])
X_val = val_df.drop(columns=[TARGET_COL, ID_COL])
X_test = test_df.drop(columns=[TARGET_COL, ID_COL])

y_train = train_df[TARGET_COL].astype("int8")
y_val = val_df[TARGET_COL].astype("int8")
y_test = test_df[TARGET_COL].astype("int8")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)


## 7.6 Inspect Target Distribution

Before choosing evaluation metrics, we need to understand class balance.

For binary classification:

- Class `0` = no purchase in the next 30 days.
- Class `1` = at least one purchase in the next 30 days.

### Why this matters

If the target is imbalanced, accuracy can become misleading.

For example:

```text
95% class 0
 5% class 1
```

A model predicting class 0 for every customer achieves:

```text
95% accuracy
```

while completely failing to identify purchasers.

Therefore, we will use:

- Precision
- Recall
- F1
- ROC-AUC
- PR-AUC

in addition to accuracy.


In [ ]:
def target_distribution(y, name):
    counts = y.value_counts().sort_index()
    percentages = y.value_counts(normalize=True).sort_index() * 100

    result = pd.DataFrame({
        "count": counts,
        "percentage": percentages
    })

    result.index.name = "target"
    print(name)
    display(result)

target_distribution(y_train, "Training target distribution")
target_distribution(y_val, "Validation target distribution")
target_distribution(y_test, "Test target distribution")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

train_counts = y_train.value_counts().sort_index()

ax.bar(
    train_counts.index.astype(str),
    train_counts.values
)

ax.set_xlabel("Target")
ax.set_ylabel("Number of Customers")
ax.set_title("Training Target Distribution")

plt.show()


## 7.7 Load Phase 6 Preprocessing Pipeline

The preprocessing pipeline contains learned transformations such as:

- numerical medians,
- scaling statistics,
- categorical vocabularies.

We load the pipeline rather than creating a new preprocessing object.

### Critical rule

```text
TRAIN
  ↓
fit_transform()

VALIDATION
  ↓
transform()

TEST
  ↓
transform()
```

The validation and test sets must never be used to fit preprocessing parameters.


In [ ]:
preprocessor = joblib.load(STANDARD_PREPROCESSOR_PATH)

print("Loaded:", STANDARD_PREPROCESSOR_PATH)
print(preprocessor)


## 7.8 Transform the Data

The Logistic Regression model will operate on the processed feature matrix produced in Phase 6.

The standard representation contains:

- imputed and standardized numerical features,
- one-hot encoded categorical features.

We keep the sparse representation to avoid unnecessary memory consumption.


In [ ]:
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed train:", X_train_processed.shape)
print("Processed validation:", X_val_processed.shape)
print("Processed test:", X_test_processed.shape)

print("Sparse:", sp.issparse(X_train_processed))


## 7.9 Processed Matrix Sanity Checks

In [ ]:
assert X_train_processed.shape[1] == X_val_processed.shape[1]
assert X_train_processed.shape[1] == X_test_processed.shape[1]

assert len(y_train) == X_train_processed.shape[0]
assert len(y_val) == X_val_processed.shape[0]
assert len(y_test) == X_test_processed.shape[0]

if sp.issparse(X_train_processed):
    assert not np.isnan(X_train_processed.data).any()
else:
    assert not np.isnan(X_train_processed).any()

print("All processed-matrix checks passed.")


# 7.10 Baseline 1 — Dummy Classifier

The first model is deliberately simple.

We use:

```text
DummyClassifier(strategy="prior")
```

It predicts based on the class distribution observed in the training data.

This is a useful reference because it answers:

> How well can we perform without learning meaningful relationships between customer features and the target?

Every future model should be compared against this baseline.


In [ ]:
dummy_model = DummyClassifier(
    strategy="prior"
)

dummy_model.fit(X_train_processed, y_train)

dummy_train_pred = dummy_model.predict(X_train_processed)
dummy_val_pred = dummy_model.predict(X_val_processed)

dummy_train_prob = dummy_model.predict_proba(X_train_processed)[:, 1]
dummy_val_prob = dummy_model.predict_proba(X_val_processed)[:, 1]

print("Dummy model trained successfully.")


## 7.11 Evaluation Metrics

We define one reusable evaluation function.

### Metrics

#### Accuracy

\[
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
\]

#### Precision

\[
Precision = \frac{TP}{TP + FP}
\]

Among predicted purchasers, how many actually purchased?

#### Recall

\[
Recall = \frac{TP}{TP + FN}
\]

Among actual purchasers, how many did we identify?

#### F1-score

\[
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
\]

F1 balances precision and recall.

#### ROC-AUC

Measures ranking quality across classification thresholds.

#### PR-AUC

Also called Average Precision.

It is particularly useful when the positive class is relatively rare because it focuses on precision-recall behavior.


In [ ]:
def evaluate_binary_classifier(
    y_true,
    y_pred,
    y_prob,
    model_name,
    split_name
):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    metrics = {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    return metrics


dummy_results = pd.DataFrame([
    evaluate_binary_classifier(
        y_train,
        dummy_train_pred,
        dummy_train_prob,
        "DummyClassifier",
        "train"
    ),
    evaluate_binary_classifier(
        y_val,
        dummy_val_pred,
        dummy_val_prob,
        "DummyClassifier",
        "validation"
    )
])

display(dummy_results)


## 7.12 Inspect Dummy Confusion Matrix

In [ ]:
dummy_cm = confusion_matrix(
    y_val,
    dummy_val_pred,
    labels=[0, 1]
)

dummy_cm_df = pd.DataFrame(
    dummy_cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

display(dummy_cm_df)


# 7.13 Baseline 2 — Logistic Regression

Logistic Regression is our first real machine-learning model.

It is an excellent baseline because it is:

- simple,
- fast,
- interpretable,
- strong for linearly separable relationships,
- compatible with sparse one-hot encoded data,
- useful for diagnosing underfitting.

The model estimates:

\[
P(y=1|x) = \sigma(w^Tx+b)
\]

where the sigmoid function is:

\[
\sigma(z)=\frac{1}{1+e^{-z}}
\]

The output is a probability between 0 and 1.

We then convert the probability into a class prediction using a threshold.


In [ ]:
logistic_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="liblinear",
    max_iter=1000,
    random_state=RANDOM_STATE
)

logistic_model.fit(X_train_processed, y_train)

print("Logistic Regression trained successfully.")


## 7.14 Generate Predictions

We generate both:

- class predictions,
- probability predictions.

Probability predictions are essential for:

- ROC curves,
- Precision-Recall curves,
- threshold tuning,
- ranking customers by purchase likelihood.


In [ ]:
log_train_pred = logistic_model.predict(X_train_processed)
log_val_pred = logistic_model.predict(X_val_processed)

log_train_prob = logistic_model.predict_proba(
    X_train_processed
)[:, 1]

log_val_prob = logistic_model.predict_proba(
    X_val_processed
)[:, 1]

print("Predictions generated.")


## 7.15 Evaluate Logistic Regression

We evaluate on both training and validation data.

The training score tells us how well the model fits the data it learned from.

The validation score tells us how well the learned relationships generalize to a future time period.

The difference between them is especially important for bias–variance analysis.


In [ ]:
logistic_results = pd.DataFrame([
    evaluate_binary_classifier(
        y_train,
        log_train_pred,
        log_train_prob,
        "LogisticRegression",
        "train"
    ),
    evaluate_binary_classifier(
        y_val,
        log_val_pred,
        log_val_prob,
        "LogisticRegression",
        "validation"
    )
])

display(logistic_results)


## 7.16 Baseline Comparison

We now compare the naive baseline with Logistic Regression.

The most important question is not:

> "Is the accuracy high?"

Instead:

> **"Does Logistic Regression meaningfully outperform the DummyClassifier, especially on PR-AUC, recall, precision and F1?"**

If yes, the engineered customer behavior features contain predictive signal.


In [ ]:
baseline_results = pd.concat(
    [dummy_results, logistic_results],
    ignore_index=True
)

display(
    baseline_results[
        [
            "model",
            "split",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc"
        ]
    ]
)


## 7.17 Validation Confusion Matrix — Logistic Regression

The confusion matrix gives a more concrete understanding of model behavior.

```text
                 Predicted
               0          1
Actual 0      TN         FP
Actual 1      FN         TP
```

For this problem:

- **TP:** customer was predicted to purchase and actually purchased.
- **FP:** customer was predicted to purchase but did not.
- **FN:** customer purchased but model missed them.
- **TN:** customer did not purchase and model correctly predicted no purchase.


In [ ]:
log_cm = confusion_matrix(
    y_val,
    log_val_pred,
    labels=[0, 1]
)

log_cm_df = pd.DataFrame(
    log_cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

display(log_cm_df)


In [ ]:
tn, fp, fn, tp = log_cm.ravel()

print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives  :", tp)


# 7.18 Bias–Variance Analysis

Bias and variance describe two important sources of model error.

### High bias — underfitting

Typical pattern:

```text
Training performance:    low
Validation performance:  low
```

The model is too simple to capture the underlying relationships.

### High variance — overfitting

Typical pattern:

```text
Training performance:    very high
Validation performance:  much lower
```

The model fits training-specific patterns that do not generalize.

### Good generalization

Typical pattern:

```text
Training performance:    strong
Validation performance:  similarly strong
```

For this project, we will inspect the train-validation gap across several metrics rather than relying on a single score.


In [ ]:
bias_variance = logistic_results.pivot(
    index="model",
    columns="split",
    values=["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
)

display(bias_variance)


In [ ]:
train_metrics = logistic_results[
    logistic_results["split"] == "train"
].iloc[0]

val_metrics = logistic_results[
    logistic_results["split"] == "validation"
].iloc[0]

gap_report = pd.DataFrame({
    "metric": [
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc"
    ],
    "train": [
        train_metrics["accuracy"],
        train_metrics["precision"],
        train_metrics["recall"],
        train_metrics["f1"],
        train_metrics["roc_auc"],
        train_metrics["pr_auc"]
    ],
    "validation": [
        val_metrics["accuracy"],
        val_metrics["precision"],
        val_metrics["recall"],
        val_metrics["f1"],
        val_metrics["roc_auc"],
        val_metrics["pr_auc"]
    ]
})

gap_report["train_minus_validation"] = (
    gap_report["train"] - gap_report["validation"]
)

display(gap_report)


## 7.19 Interpreting the Bias–Variance Results

Use the following interpretation framework.

### Case A — Both train and validation are weak

Likely issue:

> **High bias / underfitting**

Potential next steps:

- nonlinear models,
- richer features,
- interaction features,
- boosting,
- increased model complexity.

### Case B — Training is much better than validation

Likely issue:

> **High variance / overfitting**

Potential next steps:

- stronger regularization,
- simpler models,
- feature reduction,
- better temporal validation,
- hyperparameter tuning.

### Case C — Both are strong and close

This is a good baseline.

Later models should improve validation performance without creating a large generalization gap.

> **Do not decide the diagnosis using arbitrary thresholds alone.** The actual metric values and business objective matter.


# 7.20 ROC Curve

The ROC curve plots:

- True Positive Rate (Recall)
- against False Positive Rate

over different classification thresholds.

ROC-AUC summarizes ranking performance across thresholds.

A random classifier has approximately:

```text
ROC-AUC = 0.50
```

Higher values indicate better ranking ability.


In [ ]:
dummy_fpr, dummy_tpr, _ = roc_curve(
    y_val,
    dummy_val_prob
)

log_fpr, log_tpr, _ = roc_curve(
    y_val,
    log_val_prob
)

dummy_auc = roc_auc_score(y_val, dummy_val_prob)
log_auc = roc_auc_score(y_val, log_val_prob)

plt.figure(figsize=(9, 6))

plt.plot(
    dummy_fpr,
    dummy_tpr,
    label=f"Dummy (AUC={dummy_auc:.4f})"
)

plt.plot(
    log_fpr,
    log_tpr,
    label=f"Logistic Regression (AUC={log_auc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Validation Set")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# 7.21 Precision–Recall Curve

For an imbalanced purchase prediction problem, the Precision–Recall curve can be especially informative.

It shows:

- Precision
- Recall

for different probability thresholds.

A model can often increase recall by accepting more false positives, while increasing precision generally requires becoming more selective.

This trade-off will become important when selecting the operating threshold.


In [ ]:
dummy_precision, dummy_recall, _ = precision_recall_curve(
    y_val,
    dummy_val_prob
)

log_precision, log_recall, log_thresholds = precision_recall_curve(
    y_val,
    log_val_prob
)

plt.figure(figsize=(9, 6))

plt.plot(
    dummy_recall,
    dummy_precision,
    label="Dummy"
)

plt.plot(
    log_recall,
    log_precision,
    label=f"Logistic Regression (AP={average_precision_score(y_val, log_val_prob):.4f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve — Validation Set")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# 7.22 Threshold Analysis

Logistic Regression normally uses:

```text
threshold = 0.50
```

Meaning:

```text
P(purchase) >= 0.50 → class 1
P(purchase) <  0.50 → class 0
```

But 0.50 is not necessarily the best threshold for this business problem.

For example:

### Lower threshold

More customers are predicted as likely purchasers.

This generally:

- increases recall,
- decreases precision.

### Higher threshold

Fewer customers are predicted as purchasers.

This generally:

- increases precision,
- decreases recall.

Therefore, we should inspect multiple thresholds rather than blindly using 0.50.


In [ ]:
thresholds = np.arange(
    0.10,
    0.91,
    0.05
)

threshold_rows = []

for threshold in thresholds:
    pred = (log_val_prob >= threshold).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, pred, zero_division=0
        ),
        "accuracy": accuracy_score(
            y_val, pred
        )
    })

threshold_results = pd.DataFrame(threshold_rows)

display(threshold_results)


## 7.23 Visualize Threshold Trade-off

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    threshold_results["threshold"],
    threshold_results["precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["recall"],
    marker="o",
    label="Recall"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["f1"],
    marker="o",
    label="F1"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Threshold vs Precision / Recall / F1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 7.24 Select a Validation Threshold by F1

For this baseline experiment, we select the threshold that maximizes validation F1.

### Important

This is **not** the final production threshold.

It is only a baseline threshold-selection experiment.

Later, we can choose a threshold based on the actual business cost of:

- contacting a non-buyer,
- missing a potential buyer,
- recommendation capacity,
- campaign budget.

The test set must remain untouched while selecting the threshold.


In [ ]:
best_threshold_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_threshold_row["threshold"])

print("Best validation F1 threshold:", best_threshold)
display(best_threshold_row.to_frame("value"))


## 7.25 Evaluate Logistic Regression at the Selected Threshold

We now compare:

- default threshold = 0.50
- validation-F1 threshold

This shows why probability outputs and threshold tuning matter.


In [ ]:
default_threshold = 0.50

default_pred = (log_val_prob >= default_threshold).astype(int)
tuned_pred = (log_val_prob >= best_threshold).astype(int)

default_metrics = evaluate_binary_classifier(
    y_val,
    default_pred,
    log_val_prob,
    "LogisticRegression_Threshold_0.50",
    "validation"
)

tuned_metrics = evaluate_binary_classifier(
    y_val,
    tuned_pred,
    log_val_prob,
    f"LogisticRegression_Threshold_{best_threshold:.2f}",
    "validation"
)

threshold_comparison = pd.DataFrame([
    default_metrics,
    tuned_metrics
])

display(
    threshold_comparison[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc"
        ]
    ]
)


# 7.26 Model Coefficient Analysis

Logistic Regression is useful because its coefficients provide an initial interpretation of feature direction.

For a coefficient:

- positive value → higher feature value is associated with higher predicted log-odds of purchase,
- negative value → lower predicted log-odds of purchase.

Because we used standardized numerical features and one-hot encoded categorical features, coefficient magnitudes can provide useful directional insight.

However:

> **Coefficient magnitude is not the same thing as causal importance.**

Correlated features can also distribute their effects across multiple coefficients.


In [ ]:
feature_names = preprocessor.get_feature_names_out()

coefficients = logistic_model.coef_[0]

coefficient_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": np.abs(coefficients)
})

top_positive = (
    coefficient_df
    .sort_values("coefficient", ascending=False)
    .head(20)
)

top_negative = (
    coefficient_df
    .sort_values("coefficient", ascending=True)
    .head(20)
)

print("Top positive coefficients:")
display(top_positive)

print("Top negative coefficients:")
display(top_negative)


## 7.27 Compare Training and Validation Performance

A compact visualization helps us inspect the generalization gap.

We focus on:

- F1
- ROC-AUC
- PR-AUC

These are more informative than accuracy alone for this problem.


In [ ]:
plot_metrics = ["f1", "roc_auc", "pr_auc"]

metric_values_train = [
    logistic_results.loc[
        logistic_results["split"] == "train", metric
    ].iloc[0]
    for metric in plot_metrics
]

metric_values_val = [
    logistic_results.loc[
        logistic_results["split"] == "validation", metric
    ].iloc[0]
    for metric in plot_metrics
]

x = np.arange(len(plot_metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))

ax.bar(
    x - width / 2,
    metric_values_train,
    width,
    label="Train"
)

ax.bar(
    x + width / 2,
    metric_values_val,
    width,
    label="Validation"
)

ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in plot_metrics])
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_title("Logistic Regression — Train vs Validation")
ax.legend()

plt.show()


# 7.28 Save Baseline Results

We save:

- baseline metric table,
- threshold analysis,
- coefficient table.

These artifacts will be useful when comparing Phase 8 models.


In [ ]:
BASELINE_RESULTS_PATH = RESULTS_DIR / "phase7_baseline_results.csv"
THRESHOLD_RESULTS_PATH = RESULTS_DIR / "phase7_threshold_results.csv"
COEFFICIENT_RESULTS_PATH = RESULTS_DIR / "phase7_logistic_coefficients.csv"
GAP_RESULTS_PATH = RESULTS_DIR / "phase7_bias_variance_gap.csv"

baseline_results.to_csv(
    BASELINE_RESULTS_PATH,
    index=False
)

threshold_results.to_csv(
    THRESHOLD_RESULTS_PATH,
    index=False
)

coefficient_df.to_csv(
    COEFFICIENT_RESULTS_PATH,
    index=False
)

gap_report.to_csv(
    GAP_RESULTS_PATH,
    index=False
)

print("Saved:")
print(BASELINE_RESULTS_PATH)
print(THRESHOLD_RESULTS_PATH)
print(COEFFICIENT_RESULTS_PATH)
print(GAP_RESULTS_PATH)


# 7.29 Save Baseline Logistic Regression Model

We save the fitted model separately from the preprocessing pipeline.

At inference time, the workflow will be:

```text
Raw customer features
        ↓
Phase 6 preprocessing
        ↓
Processed feature matrix
        ↓
Phase 7 Logistic Regression
        ↓
Purchase probability
        ↓
Classification threshold
        ↓
Purchase / No Purchase
```


In [ ]:
LOGISTIC_MODEL_PATH = MODELS_DIR / "logistic_regression_baseline_phase7.joblib"
THRESHOLD_PATH = MODELS_DIR / "logistic_regression_threshold_phase7.json"

joblib.dump(
    logistic_model,
    LOGISTIC_MODEL_PATH
)

with open(THRESHOLD_PATH, "w") as f:
    json.dump(
        {
            "default_threshold": default_threshold,
            "validation_f1_threshold": best_threshold
        },
        f,
        indent=4
    )

print("Saved model:", LOGISTIC_MODEL_PATH)
print("Saved thresholds:", THRESHOLD_PATH)


# 7.30 Optional Test Evaluation — Final Baseline Check

The test set was **not used** for:

- preprocessing fitting,
- model fitting,
- threshold selection,
- hyperparameter selection.

Therefore, after all Phase 7 baseline decisions are finalized, we can perform one final test evaluation.

This gives an unbiased estimate of how the baseline generalizes to the final future period.

### Important

Once the test score is reported, we should not repeatedly tune the model against the test set. Doing so effectively turns the test set into a validation set.


In [ ]:
# Transform test probabilities using the already-fitted baseline model.
log_test_prob = logistic_model.predict_proba(
    X_test_processed
)[:, 1]

# Evaluate at the threshold selected using VALIDATION only.
log_test_pred = (
    log_test_prob >= best_threshold
).astype(int)

test_baseline_metrics = pd.DataFrame([
    evaluate_binary_classifier(
        y_test,
        log_test_pred,
        log_test_prob,
        f"LogisticRegression_FinalThreshold_{best_threshold:.2f}",
        "test"
    )
])

display(
    test_baseline_metrics[
        [
            "model",
            "split",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc"
        ]
    ]
)


## 7.31 Save Final Test Benchmark

This test result is the baseline benchmark against which future models can be compared.

If Phase 8 produces a model with better validation performance, we will eventually evaluate the selected model on the test set and compare it with this benchmark.


In [ ]:
TEST_RESULTS_PATH = RESULTS_DIR / "phase7_test_baseline_results.csv"

test_baseline_metrics.to_csv(
    TEST_RESULTS_PATH,
    index=False
)

print("Saved:", TEST_RESULTS_PATH)


# 7.32 Final Phase 7 Model Card

## Model

**Logistic Regression**

### Input

Customer-level engineered behavioral and demographic features from Phase 5.

### Preprocessing

Phase 6 standard preprocessing:

- median numerical imputation,
- missing indicators,
- numerical standardization,
- categorical `"Unknown"` imputation,
- one-hot encoding,
- unknown-category handling.

### Training

Temporal training set only.

### Validation

Future temporal validation set.

### Test

Held-out future temporal test set.

### Baseline comparator

`DummyClassifier(strategy="prior")`

### Primary metrics

For this project, pay particular attention to:

1. **PR-AUC**
2. **Recall**
3. **Precision**
4. **F1**
5. **ROC-AUC**

Accuracy is reported but should not be the only metric used.

### Threshold

The default threshold is 0.50.

A separate validation-selected F1 threshold is also evaluated.

---

# What we learned

After running the notebook, answer these questions:

1. How imbalanced is the target?
2. How much does Logistic Regression beat the DummyClassifier?
3. Is PR-AUC meaningfully above the baseline?
4. Is recall sufficiently high?
5. Is there a large train-validation gap?
6. Does the model appear biased or high variance?
7. Does changing the threshold materially improve F1?
8. Which customer features have the strongest positive/negative coefficients?
9. Does the final test performance resemble validation performance?

These observations should be recorded in the project's experiment log.


# 7.33 Phase 7 Deliverables

After executing the notebook, the project should contain:

```text
models/
├── preprocessor_standard_phase6.joblib
├── logistic_regression_baseline_phase7.joblib
└── logistic_regression_threshold_phase7.json

results/
├── phase7_baseline_results.csv
├── phase7_threshold_results.csv
├── phase7_logistic_coefficients.csv
├── phase7_bias_variance_gap.csv
└── phase7_test_baseline_results.csv
```

---

# Phase 8 Preview — Multiple ML Algorithms

Now that we have a benchmark, the next phase will compare several model families.

Expected candidates:

```text
Logistic Regression
        │
        ├── Linear / regularized baseline
        │
        ▼
Decision Tree
        │
        ▼
Random Forest
        │
        ▼
Extra Trees
        │
        ▼
Gradient Boosting
        │
        ▼
XGBoost / LightGBM / CatBoost
```

We will compare:

- linear models,
- bagging,
- boosting,
- tree-based models,

while maintaining the **same temporal validation protocol**.

The key goal of Phase 8 is:

> **Determine which model family can capture nonlinear customer purchase behavior better than the Phase 7 baseline without unacceptable overfitting.**
